# Energy system sankey exercise

In this tutorial, we want to create an energy consumption Sankey diagram.

We will be using one country in Europe so that we can compare using IEA data with using Eurostat data for the same country.


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from floweaver import (
    Bundle,
    Dataset,
    Elsewhere,
    Partition,
    ProcessGroup,
    SankeyDefinition,
    Waypoint,
    weave,
)

# Get a recently available year. Usually only two years ago is reliably available.
YEAR = 2023
COUNTRY = "Germany"

## Data access and processing

The first step is to download the free [IEA energy balance highlight dataset](https://www.iea.org/data-and-statistics/data-product/world-energy-balances-highlights). 
If you are connected to the University network (via VPN or eduroam), you should be able to access it directly.
Otherwise, you will need to create a free account to access it.

Once downloaded, you need to (a) select your country and year of interest (defined above) and (b) process the data from the format:

```
| Product | Flow |
```

into the format:

```
| source | target | flow | value |
```

In [ ]:
# Load data
df = pd.read_excel(
    "~/Downloads/World Energy Balances Highlights 2025.xlsx",  ## UPDATE THE EXCEL FILE PATH TO WHERE YOU HAVE IT DOWNLOADED
    sheet_name="TimeSeries_1971-2024",
    skiprows=1,  # the first row in the Excel isn't data at all, but rather a header that we don't need
    usecols="A:C,G:BH",  # there are some hidden columns in the original file that we don't want
)
df = df.set_index(["Country", "Product", "Flow"])

# Select data for country and year of interest
df_filtered = df.xs(COUNTRY, level="Country").loc[:, str(YEAR)]
# coerce the data to numeric since some may have been parsed as strings or are missing values described by "-" or similar
df_filtered = pd.to_numeric(df_filtered, errors="coerce")

display(df_filtered.unstack("Flow"))

The data requires splitting up and re-combining.

We have our `flows` clearly defined in the `Product` column, but our nodes in the Sankey (our `sources` and `targets`) are mixed together within the `Flow` column.

In this column, we have primary _sources_ of energy (imports and domestic production), intermediate _conversion_ steps (oil refinement and electricity generation), and _end-uses_ (industry, residential, etc. sectors + export).
We also have some extra data we _don't_ need (totals) and we're missing data we _do_ need (rejected energy from conversion steps).

All of this we have to handle in our processing step. 
First, we clearly define our breakdowns.

In [ ]:
flows = [
    "Coal, peat and oil shale",
    "Crude, NGL and feedstocks",
    "Natural gas",
    "Nuclear",
    "Oil products",
    "Renewables and waste",
    "Heat",
    "Electricity",
]

sources = ["Imports (PJ)", "Production (PJ)"]
uses = [
    "Commercial and public services (PJ)",
    "Industry (PJ)",
    "Other final consumption (PJ)",
    "Residential (PJ)",
    "Transport (PJ)",
    "Exports (PJ)",
]
intermediate_processes = [
    "Oil refineries, transformation (PJ)",
    "Electricity, CHP and heat plants (PJ)",
]

In [ ]:
# Here's our totals going into our end uses, which we can use to add some context to our diagram
end_use_totals = df_filtered.unstack()[uses].sum().abs()

display(end_use_totals)

In [ ]:
data = []
for flow in df_filtered.index.levels[0]:
    if flow not in flows:
        continue

    flow_df = df_filtered.xs(flow, level=0)
    no_totals_flow_df = flow_df.filter(regex="^((?!Total).)*$", axis=0).where(
        lambda x: x != 0
    )

    if no_totals_flow_df.empty:
        continue

    targets_ = no_totals_flow_df.loc[uses + intermediate_processes].dropna().abs()
    # The IEA datasets use negative values for flows into an intermediate process (targets) or for exports.
    # So we can identify flows out of intermediate processes (sources) by ignoring negative values in the dataset.
    sources_ = (
        no_totals_flow_df.loc[sources + intermediate_processes]
        .where(lambda x: x > 0)
        .dropna()
    )
    if set(targets_.index).intersection(sources_.index):
        targets_ = targets_.drop(sources_.index, errors="ignore")
    # We don't actually know how much of each source goes to each target,
    # so we will assume that the flow from each source is distributed across targets in proportion to the size of the target.
    targets_ratio = targets_ / targets_.sum()

    # Now let's build our new rows of data, combining all possible source-target pairs for this flow with the appropriate flow value.
    for src, value in sources_.items():
        for trgt, trgt_value in targets_.items():
            val = abs(value * targets_ratio.loc[trgt])
            # We remove the "(PJ)" suffix from the initial or intermediate labels since it's particularly useful,
            # and we add the total flow into each end use for context
            if trgt in end_use_totals.index:
                trgt_label = (
                    f"{trgt.replace(' (PJ)', '')} ({end_use_totals.loc[trgt]:,.0f} PJ)"
                )
            else:
                trgt_label = trgt.replace(" (PJ)", "")
            data.append(
                {
                    "source": src.replace(" (PJ)", ""),
                    "target": trgt_label,
                    "flow": flow,
                    "value": val,
                }
            )
data_df = pd.DataFrame(data)
display(data_df)

In [ ]:
# Since we've updated our source and target labels, we need to update our list of uses to match the new target labels in our data
new_uses = [
    f"{use.replace(' (PJ)', '')} ({end_use_totals.loc[use]:,.0f} PJ)" for use in uses
]
new_sources = [source.replace(" (PJ)", "") for source in sources]
new_intermediate_processes = [
    process.replace(" (PJ)", "") for process in intermediate_processes
]

In [ ]:
# Now we need to add rejected energy as flows from intermediate processes to a new "Rejected" node.
# We calculate this as the difference between the total flow into the intermediate process and the total flow out of the intermediate process.
generated = (
    data_df[data_df.source.isin(new_intermediate_processes)]
    .groupby("source")
    .value.sum()
)
input = (
    data_df[data_df.target.isin(new_intermediate_processes)]
    .groupby("target")
    .value.sum()
)
# In some cases, we get negative rejected energy; we don't want that.
rejected = (input - generated).where(lambda x: x > 0).dropna()
rejected_data_rows = (
    rejected.rename_axis("source")
    .to_frame()
    .reset_index()
    .assign(target="Rejected", flow="Rejected")
)
data_df = pd.concat([data_df, rejected_data_rows])
display(data_df)

In [ ]:
# Before we continue, we will define a color palette for our flows, which we will use later when we create our Sankey diagram.

palette = {
    "Coal, peat and oil shale": "#383838",
    "Crude, NGL and feedstocks": "#A8610B",
    "Natural gas": "#1E90FF",
    "Nuclear": "#FFD700",
    "Oil products": "#000000",
    "Renewables and waste": "#228B22",
    "Heat": "#A52A2A",
    "Electricity": "#AA15BA",
    "Rejected": "#696969",
}

### FloWeaver Sankey diagram building

With our nicely formatted table, we're ready to generate our Sankey.

We want to follow our various energy carriers through the system, so we will use colour to track them as well as distinguishing them within our primary sources.

In [ ]:
nodes = {
    # Inputs to the system, partitioned by flow type
    "imports": ProcessGroup(
        ["Imports"], partition=Partition.Simple("flow", flows), title="Imports"
    ),
    "domestic": ProcessGroup(
        ["Production"],
        partition=Partition.Simple("flow", flows),
        title="Domestic Production",
    ),
    # Intermediate processes
    "electricity": ProcessGroup(
        ["Electricity, CHP and heat plants"], title="Electricity Generation"
    ),
    "oil": ProcessGroup(["Oil refineries, transformation"], title="Oil refinery"),
    # Final uses of energy
    "uses": ProcessGroup(new_uses, partition=Partition.Simple("target", new_uses)),
    # As in the advanced floweaver tutorial, we will use an Elsewhere node to group all rejected energy
    # We partition it by the intermediate process so we can more easily see how much energy is rejected from each process
    "rejected": Waypoint(
        title=f"Rejected ({rejected.sum():,.0f} PJ)",
        partition=Partition.Simple("source", new_intermediate_processes),
    ),
}

ordering = [
    # Imports above domestic production
    [["imports"], ["domestic"], []],
    # oil refining before electricity since some refined oil goes to electricity generation
    [["oil"], []],
    [[], ["electricity"]],
    # Rejected energy below uses
    [["uses"], ["rejected"]],
]

bundles = [
    Bundle("imports", "electricity"),
    Bundle("imports", "oil"),
    Bundle("imports", "uses"),
    Bundle("domestic", "electricity"),
    Bundle("domestic", "oil"),
    Bundle("domestic", "uses"),
    Bundle("oil", "electricity"),
    Bundle("oil", "uses"),
    Bundle("oil", Elsewhere, waypoints=["rejected"]),
    Bundle("electricity", "uses"),
    Bundle("electricity", Elsewhere, waypoints=["rejected"]),
]

In [ ]:
dataset = Dataset(data_df)
flow_partition = dataset.partition("flow")
sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=flow_partition)
weave(sdd, dataset, palette=palette).to_widget(
    width=1200,
    height=450,
    margins=dict(left=180, right=350),
    debugging=True,
    # By uncommenting this and setting a format string, you can visualise flow quantities as labels.
    # link_label_format=",.0f",
)

### Plotly Sankey diagram building

In [ ]:
# Create helper function to convert flow data into plotly-compatible format
def get_sankey_data(flows: pd.DataFrame) -> tuple[list, pd.DataFrame]:
    """Helper function to convert flow data into plotly-compatible sankey data.

    Args:
        flows (pd.DataFrame): source-target-value data
    Returns:
        tuple[list, pd.DataFrame]:
            1. List of unique node labels,
            2. DataFrame with source_idx, target_idx, and value columns
    """
    # Get all unique nodes
    all_nodes = list(pd.concat([flows["source"], flows["target"]]).unique())
    node_dict = {node: idx for idx, node in enumerate(all_nodes)}

    # Map flows to node indices
    flows = flows.copy()
    flows["source_idx"] = flows["source"].map(node_dict)
    flows["target_idx"] = flows["target"].map(node_dict)

    return all_nodes, flows


all_nodes, sankey_flows = get_sankey_data(data_df)
print(f"Number of unique nodes: {len(all_nodes)}")
print(f"Number of flows: {len(sankey_flows)}")

In [ ]:
def hex_to_rgba(hex_color: str, *, opacity: float = 1) -> tuple:
    """Convert a hex colour code into an RGB colour with an alpha (opacity) component."""
    hex_color = hex_color.lstrip("#")
    if len(hex_color) == 3:
        hex_color = hex_color * 2
    return f"rgba({int(hex_color[0:2], 16)}, {int(hex_color[2:4], 16)}, {int(hex_color[4:6], 16)}, {opacity})"


colours = sankey_flows["flow"].map(palette).apply(hex_to_rgba, opacity=0.7)
display(colours)

In [ ]:
# Create the Sankey diagram
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                pad=15,
                thickness=20,
                label=all_nodes,
                line=dict(color="black", width=0.5),
            ),
            link=dict(
                source=sankey_flows["source_idx"],
                target=sankey_flows["target_idx"],
                value=sankey_flows["value"],
                color=colours,
            ),
        )
    ]
)

fig.update_layout(
    title_text=f"{COUNTRY} Energy Flows ({YEAR}) - IEA Data",
    font_size=10,
    height=500,
    width=900,
)

fig.show()

## Now it's your turn!

In the example data directory you will find a world energy balance.
It is also from the IEA, so follows the same logic as the energy balance highlights we processed above. 
Units are Terajoules (TJ).

Visualise the flows using a Sankey diagram with both FloWeaver and Plotly.

> I generated this table using a mix of tables available in the [IEA data browser](https://www.iea.org/data-and-statistics/data-tools/energy-statistics-data-browser) (table view).
> I have combined the "balances" with "renewables and waste" tables to get a higher granularity for non-fossil sources of energy.
> I have also already allocated each node's _type_ in the CSV, so you don't need to create tables for each of these separately.

When you have completed this exercise, post your generated diagrams up on [the miro board](https://miro.com/app/board/uXjVGKu9TZc=/).

For FloWeaver, use:

```py
weaved = weave(...).to_widget(...)
weaved.auto_save_png("outputs/floweaver-world-energy-balance.png")
```

For Plotly, use:

```py
fig.write_image("outputs/plotly-world-energy-balance.png")
```

In [ ]:
world_balance = pd.read_csv(
    "example_data/world-energy-balance-2023-TJ.csv", index_col=["node", "type"]
)
display(world_balance)